# R-Bot — Export GGUF (Colab)

Convierte el merge HF (`rbot-operator-merged`) a GGUF cuantizado para Ollama.

## Requisitos
1. Runtime → **GPU T4** (o High-RAM). La conversión usa mucha RAM.
2. En Drive: `MyDrive/rbot-industrial-ml/rbot-operator-merged/` con `config.json` y pesos:
   - un solo `model.safetensors`, **o**
   - shards `model-00001-of-00002.safetensors` … (típico en Qwen 3B)
3. Ejecuta las celdas en orden

## Salida
- `MyDrive/rbot-industrial-ml/rbot-operator-q4_k_m.gguf`

Luego en el PC:
```bash
# Copia el .gguf a ml/export/
cd ~/Documents/Proyectos/rbot-industrial/ml/export
ollama create rbot-operator -f Modelfile.rbot-operator
```


In [ ]:
# 0) Montar Drive y verificar merge (Operator v2 preferido)
# Qwen 3B suele guardar shards (model-00001-of-00002.safetensors), no un solo model.safetensors.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

ROOT = Path("/content/drive/MyDrive/rbot-industrial-ml")
CANDIDATES = [
    ROOT / "rbot-operator-merged",
    Path("/content/rbot-operator-merged"),  # si el notebook 03 dejó el merge local
    ROOT / "rbot-intent-merged",
]

def weight_files(d: Path):
    if not d.is_dir():
        return []
    return sorted(
        list(d.glob("model.safetensors"))
        + list(d.glob("model-*-of-*.safetensors"))
        + list(d.glob("*.bin"))
    )

MERGE = None
WEIGHTS = []
for c in CANDIDATES:
    w = weight_files(c)
    if (c / "config.json").exists() and w:
        MERGE = c
        WEIGHTS = w
        break

if MERGE is None:
    print("Contenido de Drive ml:", list(ROOT.iterdir()) if ROOT.exists() else "(no existe)")
    for c in CANDIDATES:
        print(f"  {c}: exists={c.exists()}", end="")
        if c.exists():
            print(" files=", [p.name for p in c.iterdir()][:20])
        else:
            print()
    raise AssertionError(
        "No encuentro merge con config.json + pesos. "
        "Re-ejecuta la última celda del notebook 03 (merge → Drive)."
    )

size_gb = round(sum(p.stat().st_size for p in WEIGHTS) / 1e9, 3)
print("OK merge:", MERGE)
print("pesos:", [p.name for p in WEIGHTS])
print("size GB:", size_gb)


In [ ]:
# 1) Clonar llama.cpp + deps
%cd /content
!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
%cd /content/llama.cpp
%pip install -q -U "numpy<2.3" sentencepiece protobuf "gguf>=0.10"
!ls convert_hf_to_gguf.py


In [ ]:
# 2) Copiar merge a disco local y convertir a GGUF F16
from pathlib import Path
import shutil

# MERGE ya definido en celda 0 (operator o intent)
LOCAL_MERGE = Path("/content/rbot-merged")
if LOCAL_MERGE.exists():
    shutil.rmtree(LOCAL_MERGE)
print("Copiando merge a /content …")
shutil.copytree(MERGE, LOCAL_MERGE)
print("OK", LOCAL_MERGE)

OUT_F16 = "/content/rbot-operator-f16.gguf"
%cd /content/llama.cpp
!python3 convert_hf_to_gguf.py /content/rbot-merged --outfile {OUT_F16} --outtype f16
assert Path(OUT_F16).exists(), "Fallo convert_hf_to_gguf"
print("F16 GB:", round(Path(OUT_F16).stat().st_size / 1e9, 3))


In [ ]:
# 3) Compilar llama-quantize y generar Q4_K_M
from pathlib import Path

%cd /content/llama.cpp
!cmake -B build -DGGML_NATIVE=OFF
!cmake --build build --target llama-quantize -j$(nproc)

OUT_Q4 = "/content/rbot-operator-q4_k_m.gguf"
!./build/bin/llama-quantize /content/rbot-operator-f16.gguf {OUT_Q4} Q4_K_M
assert Path(OUT_Q4).exists()
print("Q4 GB:", round(Path(OUT_Q4).stat().st_size / 1e9, 3))


In [ ]:
# 4) Guardar GGUF en Drive
import shutil
from pathlib import Path

dest_dir = Path("/content/drive/MyDrive/rbot-industrial-ml")
dest_dir.mkdir(parents=True, exist_ok=True)
dest = dest_dir / "rbot-operator-q4_k_m.gguf"
shutil.copy2("/content/rbot-operator-q4_k_m.gguf", dest)
print("Guardado en Drive:", dest)
print("GB:", round(dest.stat().st_size / 1e9, 3))
print()
print("En el PC:")
print("1) Descarga rbot-operator-q4_k_m.gguf a ml/export/")
print("2) cd ml/export && ollama create rbot-operator -f Modelfile.rbot-operator")
print("3) apps/api/.env -> OLLAMA_MODEL=rbot-operator")
print("4) bash ~/Documents/Proyectos/start-rbot.sh")
